In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [3]:
master = pd.read_csv("../data/processed/master_climate_data.csv")
coastline = pd.read_csv("../data/processed/coastline_risk_by_country.csv")

In [4]:
print(master.shape, coastline.shape)

(235, 10) (5, 5)


In [5]:
master.head()

,geo_pict,country,year,sea_level_anomaly_m,rainfall_anomaly_mm,temp_anomaly_c,population_growth_pct,tourist_arrivals,persons_affected,economic_loss_usd
0,FJ,Fiji,1979,NaN,4.7,NaN,NaN,NaN,NaN,NaN
1,FJ,Fiji,1980,NaN,11.9,NaN,NaN,NaN,NaN,NaN
2,FJ,Fiji,1981,NaN,4.7,NaN,NaN,NaN,NaN,NaN
3,FJ,Fiji,1982,NaN,7.7,NaN,NaN,NaN,NaN,NaN
4,FJ,Fiji,1983,NaN,-11.1,NaN,NaN,NaN,NaN,NaN


In [6]:
# Build an explicit code map instead of relying on geo_pict matching directly
code_map = {
    "FJI": "Fiji",
    "KIR": "Kiribati",
    "WSM": "Samoa",
    "TUV": "Tuvalu",
    "VUT": "Vanuatu",
}
coastline["country"] = coastline["geo_pict"].map(code_map)
print(coastline[["geo_pict", "country"]])

  geo_pict   country
0      FJI      Fiji
1      KIR  Kiribati
2      TUV    Tuvalu
3      VUT   Vanuatu
4      WSM     Samoa


In [7]:
from scipy import stats
 
def compute_trend(df, value_col, min_points=5):
    """Returns slope per country for a given indicator.
    min_points guards against fitting a 'trend' on too few data points —
    a slope from 3 points is not a trend, it's noise."""
    results = []
    for country, group in df.groupby("country"):
        g = group.dropna(subset=[value_col])
        if len(g) < min_points:
            results.append({"country": country, f"{value_col}_trend": np.nan,
                             f"{value_col}_n": len(g)})
            continue
        slope, intercept, r, p, se = stats.linregress(g["year"], g[value_col])
        results.append({"country": country, f"{value_col}_trend": slope,
                         f"{value_col}_n": len(g)})
    return pd.DataFrame(results)
 
sea_level_trend = compute_trend(master, "sea_level_anomaly_m")
temp_trend      = compute_trend(master, "temp_anomaly_c")
rainfall_trend  = compute_trend(master, "rainfall_anomaly_mm")
pop_trend       = compute_trend(master, "population_growth_pct")

In [8]:
def compute_avg_magnitude(df, value_col):
    return (
        df.groupby("country")[value_col]
        .apply(lambda x: x.abs().mean())
        .reset_index()
        .rename(columns={value_col: f"{value_col}_avg_magnitude"})
    )
 
temp_magnitude     = compute_avg_magnitude(master, "temp_anomaly_c")
rainfall_magnitude = compute_avg_magnitude(master, "rainfall_anomaly_mm")

In [9]:
tourism_latest = (
    master.dropna(subset=["tourist_arrivals"])
    .sort_values("year")
    .groupby("country")
    .tail(1)[["country", "year", "tourist_arrivals"]]
    .rename(columns={"year": "tourism_latest_year"})
)
print(tourism_latest)

      country  tourism_latest_year  tourist_arrivals
87   Kiribati                 2019           12000.0
137     Samoa                 2022           50600.0
231   Vanuatu                 2022           65000.0
45       Fiji                 2024          928938.0


In [10]:
pop_raw = pd.read_csv("../data/raw/Population_Total.csv", skiprows=4)
print(pop_raw.shape)
pop_raw.head()

(265, 71)


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,"Population, total",SP.POP.TOTL,54922.0,55578.0,56320.0,57002.0,57619.0,58190.0,...,108735.0,108908.0,109203.0,108587.0,107700.0,107310.0,107359.0,107995.0,108785.0,NaN
1,Africa Eastern and Southern,AFE,"Population, total",SP.POP.TOTL,130075728.0,133534923.0,137171659.0,140945536.0,144904094.0,149033472.0,...,640058741.0,657801085.0,675950189.0,694446100.0,713090928.0,731821393.0,750491370.0,769280888.0,788844284.0,NaN
2,Afghanistan,AFG,"Population, total",SP.POP.TOTL,9035043.0,9214083.0,9404406.0,9604487.0,9814318.0,10036008.0,...,35688935.0,36743039.0,37856121.0,39068979.0,40000412.0,40578842.0,41454761.0,42647492.0,43844111.0,NaN
3,Africa Western and Central,AFW,"Population, total",SP.POP.TOTL,97630925.0,99706674.0,101854756.0,104089175.0,106384410.0,108754449.0,...,440150152.0,451395343.0,462522286.0,473687685.0,484978794.0,496366058.0,508318102.0,520655398.0,532809933.0,NaN
4,Angola,AGO,"Population, total",SP.POP.TOTL,5231654.0,5301583.0,5354310.0,5408320.0,5464187.0,5521981.0,...,30234839.0,31297155.0,32375632.0,33451132.0,34532429.0,35635029.0,36749906.0,37885849.0,39040039.0,NaN


In [11]:
MY_CODES = ["FJI", "KIR", "WSM", "TUV", "VUT"]
pop_filtered = pop_raw[pop_raw["Country Code"].isin(MY_CODES)]
print(pop_filtered[["Country Name", "Country Code"]])

    Country Name Country Code
75          Fiji          FJI
123     Kiribati          KIR
244       Tuvalu          TUV
257      Vanuatu          VUT
259        Samoa          WSM


In [12]:
year_cols = [c for c in pop_filtered.columns if c.isdigit()]
 
pop_long = pop_filtered.melt(
    id_vars=["Country Name", "Country Code"],
    value_vars=year_cols,
    var_name="year",
    value_name="population_total"
)
pop_long["year"] = pop_long["year"].astype(int)
pop_long = pop_long.rename(columns={"Country Code": "geo_pict"})
pop_long = pop_long.dropna(subset=["population_total"])
 
print(pop_long.shape)
pop_long.head()

(330, 4)


,Country Name,geo_pict,year,population_total
0,Fiji,FJI,1960,404887.0
1,Kiribati,KIR,1960,47157.0
2,Tuvalu,TUV,1960,5598.0
3,Vanuatu,VUT,1960,64431.0
4,Samoa,WSM,1960,112490.0


In [13]:
pop_latest = (
    pop_long.sort_values("year")
    .groupby("Country Name")
    .tail(1)[["Country Name", "year", "population_total"]]
    .rename(columns={"year": "population_year", "Country Name":"country"})
)
print(pop_latest)

      country  population_year  population_total
325      Fiji             2025          933154.0
326  Kiribati             2025          136488.0
327    Tuvalu             2025            9492.0
328   Vanuatu             2025          335169.0
329     Samoa             2025          219306.0


In [15]:
code_to_country = master[["geo_pict", "country"]].drop_duplicates()

In [16]:
risk_components = code_to_country.copy()
for component_df in [sea_level_trend, temp_trend, rainfall_trend, pop_trend,
                      temp_magnitude, rainfall_magnitude, tourism_latest,
                      coastline[["country", "median_rate_m_per_yr", "pct_eroding"]]]:
    risk_components = risk_components.merge(component_df, on="country", how="left")
 
risk_components

,geo_pict,country,sea_level_anomaly_m_trend,sea_level_anomaly_m_n,temp_anomaly_c_trend,temp_anomaly_c_n,rainfall_anomaly_mm_trend,rainfall_anomaly_mm_n,population_growth_pct_trend,population_growth_pct_n,temp_anomaly_c_avg_magnitude,rainfall_anomaly_mm_avg_magnitude,tourism_latest_year,tourist_arrivals,median_rate_m_per_yr,pct_eroding
0,FJ,Fiji,0.004153,31,0.023076,36,0.096149,47,-0.027382,36,0.397222,9.731915,2024.0,928938.0,0.0060,0.492040
1,KI,Kiribati,0.003911,31,0.002046,36,-0.227000,47,-0.014033,36,0.497222,16.421277,2019.0,12000.0,-0.1570,0.641050
2,WS,Samoa,0.004516,31,0.013964,36,0.240877,47,0.006793,36,0.375000,7.993617,2022.0,50600.0,0.1010,0.367225
3,TV,Tuvalu,0.004395,31,0.012317,36,-0.200046,47,-0.083755,36,0.386111,12.804255,NaN,NaN,0.1985,0.329836
4,VU,Vanuatu,0.004597,31,0.024955,36,0.266998,47,-0.008021,36,0.386111,9.929787,2022.0,65000.0,-0.0130,0.514614


In [17]:
risk_components = risk_components.merge(pop_latest, on="country", how="left")

In [18]:
risk_components["tourism_per_capita"] = (
    risk_components["tourist_arrivals"] / risk_components["population_total"] * 100
)
print(risk_components[["country", "tourist_arrivals", "population_total", "tourism_per_capita"]])

    country  tourist_arrivals  population_total  tourism_per_capita
0      Fiji          928938.0          933154.0           99.548199
1  Kiribati           12000.0          136488.0            8.791982
2     Samoa           50600.0          219306.0           23.072784
3    Tuvalu               NaN            9492.0                 NaN
4   Vanuatu           65000.0          335169.0           19.393202


In [21]:
scaler = MinMaxScaler(feature_range=(0, 100))
has_tourism = risk_components["tourism_per_capita"].notna()
risk_components["economic_score"] = np.nan
risk_components.loc[has_tourism, "economic_score"] = scaler.fit_transform(
    risk_components.loc[has_tourism, ["tourism_per_capita"]]
).flatten()
print(risk_components[["country", "tourist_arrivals", "tourism_per_capita", "economic_score"]])

    country  tourist_arrivals  tourism_per_capita  economic_score
0      Fiji          928938.0           99.548199      100.000000
1  Kiribati           12000.0            8.791982        0.000000
2     Samoa           50600.0           23.072784       15.735343
3    Tuvalu               NaN                 NaN             NaN
4   Vanuatu           65000.0           19.393202       11.680985


In [22]:
def rank_normalize(series):
    """Converts to percentile rank (0-100), robust to outlier magnitude."""
    return series.rank(pct=True) * 100
 
for col, raw_col in [
    ("physical_sea_level_score", "sea_level_anomaly_m_trend"),
    ("physical_temp_score", "temp_anomaly_c_avg_magnitude"),
    ("physical_rainfall_score", "rainfall_anomaly_mm_avg_magnitude"),
    ("population_score", "population_growth_pct_trend"),
]:
    risk_components[col] = rank_normalize(risk_components[raw_col].fillna(risk_components[raw_col].mean()))
 
# Coastline stays inverted (lower/more negative rate = higher risk)
risk_components["physical_coastline_score"] = rank_normalize(
    -risk_components["median_rate_m_per_yr"].fillna(risk_components["median_rate_m_per_yr"].mean())
)
risk_components["physical_erosion_share_score"] = rank_normalize(risk_components["pct_eroding"])
 
risk_components.loc[has_tourism, "economic_score"] = rank_normalize(
    risk_components.loc[has_tourism, "tourism_per_capita"]
)
 
# %% [markdown]
# ## Recompute physical + composite scores with corrected components
 
# %%
physical_cols = ["physical_sea_level_score", "physical_coastline_score",
                  "physical_erosion_share_score", "physical_temp_score",
                  "physical_rainfall_score"]
risk_components["physical_score"] = risk_components[physical_cols].mean(axis=1)
 
def compute_composite(row):
    if pd.isna(row["economic_score"]):
        return 0.65 * row["physical_score"] + 0.35 * row["population_score"]
    else:
        return (0.50 * row["physical_score"]
                + 0.25 * row["population_score"]
                + 0.25 * row["economic_score"])
 
risk_components["climate_risk_index"] = risk_components.apply(compute_composite, axis=1)
 
final_ranking = risk_components[
    ["country", "climate_risk_index", "physical_score", "population_score", "economic_score"]
].sort_values("climate_risk_index", ascending=False).reset_index(drop=True)
 
print(final_ranking)

    country  climate_risk_index  physical_score  population_score  \
0   Vanuatu               69.50            74.0              80.0   
1     Samoa               63.75            40.0             100.0   
2  Kiribati               63.25            84.0              60.0   
3      Fiji               63.00            56.0              40.0   
4    Tuvalu               36.90            46.0              20.0   

   economic_score  
0            50.0  
1            75.0  
2            25.0  
3           100.0  
4             NaN  


In [24]:
import os
os.makedirs("../data/processed", exist_ok=True)
risk_components.to_csv("../data/processed/climate_risk_index_full_V1.csv", index=False)
final_ranking.to_csv("../data/processed/climate_risk_index_ranking_V1.csv", index=False)
print("Saved climate_risk_index_full.csv and climate_risk_index_ranking.csv")

Saved climate_risk_index_full.csv and climate_risk_index_ranking.csv


In [3]:
risk_components = pd.read_csv("../data/processed/climate_risk_index_full_V1.csv")

def compute_contributions(row):
    if pd.isna(row["economic_score"]):
        total = 0.65 * row["physical_score"] + 0.35 * row["population_score"]
        return pd.Series({
            "contribution_physical_pct": (0.65 * row["physical_score"]) / total * 100 if total else np.nan,
            "contribution_population_pct": (0.35 * row["population_score"]) / total * 100 if total else np.nan,
            "contribution_economic_pct": 0.0,
        })
    else:
        total = row["climate_risk_index"]
        return pd.Series({
            "contribution_physical_pct": (0.50 * row["physical_score"]) / total * 100 if total else np.nan,
            "contribution_population_pct": (0.25 * row["population_score"]) / total * 100 if total else np.nan,
            "contribution_economic_pct": (0.25 * row["economic_score"]) / total * 100 if total else np.nan,
        })

import numpy as np
contributions = risk_components.apply(compute_contributions, axis=1)
risk_components = pd.concat([risk_components, contributions], axis=1)

# Re-save canonical files, now with contributions restored on the corrected scores
final_ranking = risk_components[
    ["country", "climate_risk_index", "physical_score", "population_score", "economic_score",
     "tourism_per_capita", "contribution_physical_pct", "contribution_population_pct", "contribution_economic_pct"]
].sort_values("climate_risk_index", ascending=False).reset_index(drop=True)

risk_components.to_csv("../data/processed/climate_risk_index_full.csv", index=False)
final_ranking.to_csv("../data/processed/climate_risk_index_ranking.csv", index=False)

print(final_ranking[["country", "climate_risk_index", "contribution_physical_pct",
                      "contribution_population_pct", "contribution_economic_pct"]])

    country  climate_risk_index  contribution_physical_pct  \
0   Vanuatu               69.50                  53.237410   
1     Samoa               63.75                  31.372549   
2  Kiribati               63.25                  66.403162   
3      Fiji               63.00                  44.444444   
4    Tuvalu               36.90                  81.029810   

   contribution_population_pct  contribution_economic_pct  
0                    28.776978                  17.985612  
1                    39.215686                  29.411765  
2                    23.715415                   9.881423  
3                    15.873016                  39.682540  
4                    18.970190                   0.000000  
